# User Defined Functions (UDFs) in Databricks

**User Defined Functions (UDFs)** allow you to extend Databricks SQL and PySpark with custom logic that isn't available in built-in functions. You can write functions in SQL, Python, Scala, or Java and use them across queries and dataframes.

## Key Concepts

* **SQL UDFs**: Written in SQL, optimized by Catalyst, best performance
* **Python UDFs**: Written in Python, flexible but slower (serialization overhead)
* **Pandas UDFs**: Vectorized Python UDFs, much faster than row-at-a-time UDFs
* **Scalar UDFs**: Process one row at a time
* **Table-Valued Functions**: Return entire tables/datasets

---

## Types of UDFs

| Type | Language | Performance | Use Case |
|------|----------|-------------|----------|
| **SQL UDF** | SQL | ⚡ Fastest | Simple transformations, built-in function combinations |
| **Python UDF** | Python | 🐌 Slow | Complex logic, external libraries |
| **Pandas UDF** | Python | ⚡ Fast | Vectorized operations, ML inference |
| **Scala UDF** | Scala | ⚡ Fast | Complex logic, type safety |

---

## When to Use UDFs

✅ **Use UDFs when:**
* Built-in functions don't support your use case
* Need to reuse complex logic across queries
* Integrating with external libraries
* Standardizing business logic

❌ **Avoid UDFs when:**
* Built-in functions can do the job (always prefer built-in)
* Simple transformations (use CASE WHEN, column expressions)
* Performance is critical (UDFs can be slower)

---

Let's explore each type with examples!

In [0]:
%sql
-- Create sample data for demonstrations
CREATE OR REPLACE TEMP VIEW employees AS
SELECT 1 as emp_id, 'Alice' as name, 'alice@company.com' as email, 75000 as salary, 'Engineering' as department
UNION ALL
SELECT 2 as emp_id, 'Bob' as name, 'bob@company.com' as email, 65000 as salary, 'Sales'
UNION ALL
SELECT 3 as emp_id, 'Charlie' as name, 'charlie@company.com' as email, 80000 as salary, 'Engineering'
UNION ALL
SELECT 4 as emp_id, 'Diana' as name, 'diana@company.com' as email, 70000 as salary, 'Marketing';

SELECT * FROM employees;

In [0]:
%sql
-- SQL UDFs: Written in SQL, optimized by Catalyst optimizer
-- Best performance, no serialization overhead

-- Example 1: Simple calculation
CREATE OR REPLACE FUNCTION calculate_bonus(salary DOUBLE, performance_rating INT)
RETURNS DOUBLE
RETURN salary * performance_rating * 0.05;

-- Use the function
SELECT 
  emp_id,
  name,
  salary,
  calculate_bonus(salary, 4) as bonus_amount
FROM employees;

-- Example 2: String manipulation
CREATE OR REPLACE FUNCTION mask_email(email STRING)
RETURNS STRING
RETURN concat(left(email, 3), '***@', split(email, '@')[1]);

SELECT 
  emp_id,
  name,
  email,
  mask_email(email) as masked_email
FROM employees;

In [0]:
%sql
-- SQL UDFs with CASE statements and complex logic

-- Example 1: Salary grade calculation
CREATE OR REPLACE FUNCTION get_salary_grade(salary DOUBLE)
RETURNS STRING
RETURN CASE
  WHEN salary < 50000 THEN 'Junior'
  WHEN salary >= 50000 AND salary < 70000 THEN 'Mid-Level'
  WHEN salary >= 70000 AND salary < 90000 THEN 'Senior'
  ELSE 'Executive'
END;

SELECT 
  emp_id,
  name,
  salary,
  get_salary_grade(salary) as grade
FROM employees;

-- Example 2: Tax calculation with multiple brackets
CREATE OR REPLACE FUNCTION calculate_tax(income DOUBLE)
RETURNS DOUBLE
RETURN CASE
  WHEN income <= 50000 THEN income * 0.10
  WHEN income <= 100000 THEN 5000 + (income - 50000) * 0.20
  ELSE 15000 + (income - 100000) * 0.30
END;

SELECT 
  emp_id,
  name,
  salary,
  calculate_tax(salary) as tax_amount,
  salary - calculate_tax(salary) as after_tax_income
FROM employees;

In [0]:
# Python UDFs: More flexible but slower due to serialization
# Use when you need Python libraries or complex logic

from pyspark.sql.functions import udf
from pyspark.sql.types import StringType, DoubleType
import hashlib

# Example 1: Hash email for privacy
@udf(returnType=StringType())
def hash_email(email):
    if email:
        return hashlib.md5(email.encode()).hexdigest()
    return None

# Example 2: Advanced string formatting
@udf(returnType=StringType())
def format_name(name):
    if name:
        return name.upper().replace(' ', '_')
    return None

# Example 3: Complex calculation
@udf(returnType=DoubleType())
def calculate_bonus_python(salary, years_of_service):
    base_bonus = salary * 0.05
    tenure_multiplier = 1 + (years_of_service * 0.02)
    return base_bonus * tenure_multiplier

# Apply the UDFs
df = spark.table("employees")
df_transformed = df.select(
    "emp_id",
    "name",
    "email",
    hash_email("email").alias("hashed_email"),
    format_name("name").alias("formatted_name")
)

display(df_transformed)

In [0]:
# Pandas UDFs (Vectorized UDFs): Much faster than row-at-a-time Python UDFs
# Process data in batches using pandas Series/DataFrames

from pyspark.sql.functions import pandas_udf
import pandas as pd

# Example 1: Vectorized string transformation
@pandas_udf(StringType())
def vectorized_upper(names: pd.Series) -> pd.Series:
    return names.str.upper()

# Example 2: Vectorized calculation with multiple inputs
@pandas_udf(DoubleType())
def vectorized_bonus(salaries: pd.Series, ratings: pd.Series) -> pd.Series:
    return salaries * ratings * 0.05

# Example 3: Complex vectorized logic
@pandas_udf(StringType())
def categorize_salary(salaries: pd.Series) -> pd.Series:
    return pd.cut(
        salaries,
        bins=[0, 50000, 70000, 90000, float('inf')],
        labels=['Junior', 'Mid-Level', 'Senior', 'Executive']
    )

# Apply Pandas UDFs
df = spark.table("employees")
df_result = df.select(
    "emp_id",
    "name",
    "salary",
    vectorized_upper("name").alias("name_upper"),
    categorize_salary("salary").alias("salary_category")
)

display(df_result)

print("\n✅ Pandas UDFs are 10-100x faster than regular Python UDFs!")

In [0]:
# Register Python/Pandas UDFs so they can be used in SQL queries

from pyspark.sql.functions import udf, pandas_udf
from pyspark.sql.types import StringType, DoubleType
import pandas as pd

# Method 1: Register using spark.udf.register
def python_upper(text):
    return text.upper() if text else None

spark.udf.register("upper_udf", python_upper, StringType())

# Method 2: Register Pandas UDF
@pandas_udf(DoubleType())
def bonus_calculator(salary: pd.Series) -> pd.Series:
    return salary * 0.1

spark.udf.register("bonus_calculator", bonus_calculator)

# Now use them in SQL!
print("Registered UDFs for SQL use:\n")
spark.sql("""
    SELECT 
        emp_id,
        name,
        upper_udf(name) as name_upper,
        salary,
        bonus_calculator(salary) as bonus
    FROM employees
""").show()

print("\n✅ Python UDFs can now be used in SQL queries!")

In [0]:
%sql
-- Table-Valued Functions: Return entire result sets
-- Useful for encapsulating complex queries

-- Example 1: Get employees by department
CREATE OR REPLACE FUNCTION get_employees_by_dept(dept_name STRING)
RETURNS TABLE(emp_id INT, name STRING, email STRING, salary DOUBLE)
RETURN SELECT emp_id, name, email, salary 
       FROM employees 
       WHERE department = dept_name;

-- Use the TVF
SELECT * FROM get_employees_by_dept('Engineering');

-- Example 2: Get high earners above threshold
CREATE OR REPLACE FUNCTION get_high_earners(min_salary DOUBLE)
RETURNS TABLE(emp_id INT, name STRING, salary DOUBLE, grade STRING)
RETURN SELECT 
         emp_id, 
         name, 
         salary,
         get_salary_grade(salary) as grade
       FROM employees 
       WHERE salary >= min_salary
       ORDER BY salary DESC;

SELECT * FROM get_high_earners(70000);

In [0]:
# Performance comparison: Built-in vs Python UDF vs Pandas UDF
import time
from pyspark.sql.functions import col, upper, udf, pandas_udf
import pandas as pd

# Create a larger dataset for testing
large_df = spark.range(0, 100000).select(
    col("id"),
    col("id").cast("string").alias("text")
)

print("Performance Comparison (100K rows):\n")
print("="*60)

# 1. Built-in function (FASTEST)
start = time.time()
large_df.select(upper("text")).count()
builtin_time = time.time() - start
print(f"1. Built-in function (upper): {builtin_time:.3f} seconds")

# 2. Python UDF (SLOWEST)
@udf(returnType="string")
def python_upper_udf(text):
    return text.upper() if text else None

start = time.time()
large_df.select(python_upper_udf("text")).count()
python_udf_time = time.time() - start
print(f"2. Python UDF: {python_udf_time:.3f} seconds ({python_udf_time/builtin_time:.1f}x slower)")

# 3. Pandas UDF (FASTER)
@pandas_udf("string")
def pandas_upper_udf(texts: pd.Series) -> pd.Series:
    return texts.str.upper()

start = time.time()
large_df.select(pandas_upper_udf("text")).count()
pandas_udf_time = time.time() - start
print(f"3. Pandas UDF: {pandas_udf_time:.3f} seconds ({pandas_udf_time/builtin_time:.1f}x slower)")

print("\n" + "="*60)
print("\n⚡ Key Takeaway: Always prefer built-in functions!")
print("\u2705 If you must use Python, prefer Pandas UDFs over regular UDFs")

In [0]:
%sql
-- Managing UDFs in Databricks

-- List all functions (including UDFs) in current database
SHOW USER FUNCTIONS;

-- Describe a specific function
DESCRIBE FUNCTION calculate_bonus;

-- Show extended information
DESCRIBE FUNCTION EXTENDED mask_email;

-- Drop a function
-- DROP FUNCTION IF EXISTS calculate_bonus;

-- Create OR REPLACE to update existing function
CREATE OR REPLACE FUNCTION calculate_bonus(salary DOUBLE, performance_rating INT)
RETURNS DOUBLE
COMMENT 'Calculates employee bonus based on salary and performance rating'
RETURN salary * performance_rating * 0.05;

In [0]:
%sql
-- Unity Catalog UDFs: Governed, shareable across workspaces
-- Syntax: CREATE FUNCTION catalog.schema.function_name

-- Example: Create a UC function (if you have permissions)
-- CREATE OR REPLACE FUNCTION workspace.default.uc_calculate_bonus(
--     salary DOUBLE, 
--     rating INT
-- )
-- RETURNS DOUBLE
-- COMMENT 'Unity Catalog managed bonus calculation function'
-- RETURN salary * rating * 0.05;

-- Use UC function
-- SELECT 
--     emp_id,
--     name,
--     salary,
--     workspace.default.uc_calculate_bonus(salary, 4) as bonus
-- FROM employees;

-- Grant permissions
-- GRANT EXECUTE ON FUNCTION workspace.default.uc_calculate_bonus TO `user@company.com`;

SELECT 'Unity Catalog UDFs provide governance and sharing capabilities!' as info;

## User Defined Functions - Complete Reference

---

### UDF Types Comparison

| Type | Language | Performance | Serialization | Optimization | Use Case |
|------|----------|-------------|---------------|--------------|----------|
| **SQL UDF** | SQL | ⚡⚡⚡ Fastest | None | Catalyst | Simple logic, reusable queries |
| **Python UDF** | Python | 🐌 Slowest | Row-by-row | None | Complex Python logic, libraries |
| **Pandas UDF** | Python + Pandas | ⚡⚡ Fast | Vectorized | Partial | ML inference, vectorized ops |
| **Scala UDF** | Scala | ⚡⚡⚡ Fastest | None | Yes | Type-safe complex logic |
| **Java UDF** | Java | ⚡⚡⚡ Fast | None | Yes | Enterprise Java integration |

---

### Creating UDFs

#### SQL UDF Syntax
```sql
CREATE [OR REPLACE] FUNCTION [catalog.]schema.function_name(
    param1 TYPE,
    param2 TYPE
)
RETURNS return_type
[COMMENT 'description']
RETURN expression_or_logic;
```

#### Python UDF Syntax
```python
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

@udf(returnType=StringType())
def my_function(input_value):
    return processed_value
```

#### Pandas UDF Syntax
```python
from pyspark.sql.functions import pandas_udf
import pandas as pd

@pandas_udf(StringType())
def my_pandas_function(inputs: pd.Series) -> pd.Series:
    return processed_series
```

#### Table-Valued Function Syntax
```sql
CREATE FUNCTION function_name(param TYPE)
RETURNS TABLE(col1 TYPE, col2 TYPE)
RETURN SELECT col1, col2 FROM table WHERE condition;
```

---

### Performance Guidelines

#### Performance Ranking (Fastest to Slowest)
1. **Built-in functions** — Always use these first!
2. **SQL UDFs** — Optimized by Catalyst
3. **Scala/Java UDFs** — Native JVM, no serialization
4. **Pandas UDFs** — Vectorized, batch processing
5. **Python UDFs** — Row-by-row, serialization overhead

#### Why Python UDFs are Slow
* **Serialization overhead**: Data converted between JVM and Python
* **Row-by-row processing**: No vectorization
* **No Catalyst optimization**: Black box to Spark optimizer
* **GIL limitations**: Single-threaded Python execution

#### Optimization Tips
✅ **Prefer built-in functions** — Check if combination of built-ins works  
✅ **Use SQL UDFs when possible** — Best performance for custom logic  
✅ **Use Pandas UDFs over Python UDFs** — 10-100x faster  
✅ **Filter before UDF** — Reduce data processed by UDF  
✅ **Cache results** — If UDF output reused multiple times  

❌ **Don't use UDFs in WHERE clauses** — Prevents predicate pushdown  
❌ **Don't use UDFs in JOIN conditions** — Kills broadcast optimization  
❌ **Don't nest UDFs deeply** — Compounds performance issues  

---

### Common Use Cases

#### 1. Data Masking / Privacy
```sql
-- Mask PII data
CREATE FUNCTION mask_ssn(ssn STRING)
RETURNS STRING
RETURN concat('XXX-XX-', right(ssn, 4));
```

#### 2. Business Logic Standardization
```sql
-- Standard commission calculation
CREATE FUNCTION calculate_commission(sales DOUBLE, tier STRING)
RETURNS DOUBLE
RETURN CASE
  WHEN tier = 'Gold' THEN sales * 0.15
  WHEN tier = 'Silver' THEN sales * 0.10
  ELSE sales * 0.05
END;
```

#### 3. External API Integration
```python
# Call external service (use with caution!)
@udf(returnType=StringType())
def geocode_address(address):
    # Call geocoding API
    return coordinates
```

#### 4. ML Model Inference
```python
# Pandas UDF for batch inference
@pandas_udf(DoubleType())
def predict(features: pd.Series) -> pd.Series:
    return model.predict(features)
```

---

### Best Practices

#### Development
✅ **Start with built-in functions** — Always check first  
✅ **Use SQL UDFs for simple logic** — Best performance  
✅ **Add comments and descriptions** — `COMMENT 'description'`  
✅ **Version control UDFs** — Track changes in Git  
✅ **Test with sample data** — Validate logic before production  

#### Performance
✅ **Use Pandas UDFs for Python** — Vectorization is key  
✅ **Apply UDFs after filtering** — Process less data  
✅ **Cache results if reused** — `df.cache()`  
✅ **Monitor execution plans** — Check `EXPLAIN`  
✅ **Profile UDF performance** — Use timer decorators  

#### Production
✅ **Use Unity Catalog for governance** — Centralized management  
✅ **Grant appropriate permissions** — Least privilege  
✅ **Document parameters and return types** — Clear contracts  
✅ **Handle NULL values explicitly** — Avoid errors  
✅ **Add error handling** — Try/catch in Python UDFs  

#### Anti-Patterns
❌ **Using UDFs when built-ins work** — Slower for no reason  
❌ **Python UDFs in tight loops** — Extreme performance hit  
❌ **UDFs without NULL handling** — Runtime errors  
❌ **Complex UDFs without testing** — Production bugs  
❌ **UDFs accessing external state** — Not serializable  

---

### Alternatives to UDFs

Before writing a UDF, consider:

| Instead of... | Try... |
|---------------|--------|
| Python UDF for string ops | `regexp_extract`, `concat`, `substring` |
| Python UDF for math | `CASE WHEN`, mathematical operators |
| Python UDF for aggregation | Built-in agg functions, window functions |
| Python UDF for conditionals | `CASE WHEN`, `COALESCE`, `NVL` |
| Python UDF for arrays | Higher order functions (`transform`, `filter`) |
| Python UDF for dates | `date_add`, `datediff`, `date_format` |

---

### Unity Catalog Integration

#### Creating UC Functions
```sql
CREATE FUNCTION catalog.schema.function_name(...)
RETURNS type
RETURN expression;
```

#### Benefits of UC Functions
* **Centralized management**: One place to manage all functions
* **Access control**: GRANT/REVOKE permissions
* **Cross-workspace sharing**: Use same function across workspaces
* **Lineage tracking**: See where functions are used
* **Versioning**: Track changes over time

#### Permissions
```sql
-- Grant execute permission
GRANT EXECUTE ON FUNCTION catalog.schema.function_name TO `user@company.com`;

-- Grant to group
GRANT EXECUTE ON FUNCTION catalog.schema.function_name TO `data_analysts`;

-- Revoke permission
REVOKE EXECUTE ON FUNCTION catalog.schema.function_name FROM `user@company.com`;
```

---

### Debugging UDFs

#### Python UDF Debugging
```python
# Add logging
import logging

@udf(returnType=StringType())
def debug_udf(value):
    logging.info(f"Processing: {value}")
    try:
        result = process(value)
        return result
    except Exception as e:
        logging.error(f"Error: {e}")
        return None
```

#### SQL UDF Debugging
```sql
-- Test with sample data first
SELECT 
    input_value,
    my_function(input_value) as output
FROM (VALUES ('test1'), ('test2'), (NULL)) AS t(input_value);
```

---

### Additional Resources

* **Databricks Documentation**: Search for "User Defined Functions"
* **Pandas UDF Guide**: Look for "Pandas UDF" in Databricks docs
* **Performance Tuning**: Check Spark SQL optimization guides
* **Unity Catalog**: Learn about UC function governance